In [1]:
NOTEBOOK_ARGS = [
    # 自动识别失败时，取消注释其中一行并填写真实路径：
    # "--zip", "/kaggle/input/你的数据集/2026task2.zip",
    # "--data-dir", "/kaggle/input/你的数据集",
    "--output", "/kaggle/working",
    "--epochs", "35",
    "--batch-size", "64",
    # "--no-pretrained",  # 无法下载权重时使用，从随机参数开始训练
]


In [2]:
#!/usr/bin/env python3
"""Train a FER classifier from the original ZIP or an extracted folder; see README.md."""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import random
import time
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path, PurePosixPath

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

# These are INTERNAL model indices, NOT competition submission IDs.
CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
# Official submission IDs supplied by the user; also available when pasted into a notebook.
DEFAULT_LABEL_MAP = {'Angry': 0, 'Fear': 1, 'Happy': 2, 'Neutral': 5, 'Sad': 3, 'Surprise': 4}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def default_output_dir():
    """Save Kaggle artifacts in its working directory; retain the local default."""
    kaggle_working = Path('/kaggle/working')
    return kaggle_working if kaggle_working.is_dir() else Path('runs/fer')


def read_mapping(path):
    mapping = (DEFAULT_LABEL_MAP.copy() if path is None else
               json.loads(Path(path).read_text(encoding='utf-8-sig')))
    if set(mapping) != set(CLASSES):
        raise ValueError(f'label_map must contain exactly these keys: {CLASSES}')
    if any(type(v) is not int for v in mapping.values()):
        raise ValueError('Fill EVERY null in label_map.json with an official integer ID. '
                         'The ZIP does not specify this mapping; alphabetical IDs are unsafe.')
    if len(set(mapping.values())) != len(CLASSES):
        raise ValueError('Submission IDs must be unique across the six classes.')
    return mapping


def discover_data(input_root=Path('/kaggle/input')):
    """Find this task only; refuse to silently choose between multiple datasets."""
    archives = sorted(input_root.rglob('2026task2.zip')) if input_root.is_dir() else []
    local_zip = Path.cwd() / '2026task2.zip'
    if local_zip.is_file() and local_zip not in archives:
        archives.append(local_zip)
    if len(archives) == 1:
        return archives[0]
    if len(archives) > 1:
        raise ValueError('Multiple 2026task2.zip files found; set --zip explicitly: '
                         + ', '.join(str(p) for p in archives))
    folders = []
    templates = sorted(input_root.rglob('submission.csv')) if input_root.is_dir() else []
    for template in templates:
        root = template.parent
        for relative in ['', 'fer_data', 'fer_data/fer_data']:
            base = root / relative
            if (base / 'train').is_dir() and (base / 'test').is_dir():
                folders.append(root)
                break
    if len(folders) == 1:
        return folders[0]
    if len(folders) > 1:
        raise ValueError('Multiple extracted datasets found; set --data-dir explicitly: '
                         + ', '.join(str(p) for p in folders))
    raise ValueError('Dataset not found. Add the task data to the Notebook, or specify '
                     '--zip /path/2026task2.zip or --data-dir /path/to/folder. '
                     'The extracted folder must contain submission.csv and the train/test trees.')


@contextmanager
def open_data_source(path):
    """Present ZIP and extracted files through the same strict validation path."""
    path = Path(path)
    if path.is_dir():
        names = [p.relative_to(path).as_posix() for p in path.rglob('*') if p.is_file()
                 and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.csv'}]

        def open_member(name):
            return (path / PurePosixPath(name)).open('rb')

        yield names, open_member
    else:
        with zipfile.ZipFile(path) as z:
            yield [n for n in z.namelist() if not n.endswith('/')], z.open


def read_archive(path, include_train=True):
    """Validate the template and load grayscale pixels from ZIP or extracted data."""
    images, labels, train_ids = [], [], []
    with open_data_source(path) as (names, open_member):
        templates = [n for n in names if PurePosixPath(n).name == 'submission.csv']
        if len(templates) != 1:
            raise ValueError('Expected exactly one submission.csv template in the data source.')
        with io.TextIOWrapper(open_member(templates[0]), encoding='utf-8-sig', newline='') as f:
            reader = csv.DictReader(f)
            if reader.fieldnames != ['ID', 'Emotion']:
                raise ValueError(f'Unexpected template columns: {reader.fieldnames}')
            rows = list(reader)
        ids = [r['ID'] for r in rows]
        if not ids or any(not v for v in ids) or len(ids) != len(set(ids)):
            raise ValueError('Template IDs are empty or duplicated.')
        test_members = {}
        train_members = []
        for name in names:
            p = PurePosixPath(name)
            if p.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                continue
            if p.parent.name == 'test':
                if p.name in test_members:
                    raise ValueError(f'Duplicate test filename: {p.name}')
                test_members[p.name] = name
            elif p.parent.parent.name == 'train':
                if p.parent.name not in CLASSES:
                    raise ValueError(f'Unexpected class folder: {p.parent.name}')
                train_members.append((name, CLASSES.index(p.parent.name)))
        if set(ids) != set(test_members):
            raise ValueError('Template IDs do not exactly match test image filenames.')

        def pixels(member):
            with open_member(member) as f, Image.open(f) as im:
                if im.size != (48, 48):
                    raise ValueError(f'Expected 48x48 image: {member}, got {im.size}')
                return np.array(im.convert('L'), dtype=np.uint8)

        if include_train:
            for name, label in tqdm(sorted(train_members), desc='Load train'):
                images.append(pixels(name))
                labels.append(label)
                train_ids.append(name)
            if set(labels) != set(range(len(CLASSES))):
                raise ValueError('Training data does not contain all six classes.')
        # IMPORTANT: preserve the template order, never sort test IDs.
        tests = np.stack([pixels(test_members[i]) for i in tqdm(ids, desc='Load test')])
    train = np.stack(images) if images else None
    return train, np.asarray(labels, dtype=np.int64), train_ids, tests, ids


def clean_split(images, labels, val_ratio, seed):
    """One copy per unique pixel array; discard groups with conflicting labels."""
    groups = defaultdict(list)
    for i, pixels in enumerate(images):
        groups[hashlib.sha256(pixels.tobytes()).digest()].append(i)
    keep, conflict_groups, conflict_samples = [], 0, 0
    for group in groups.values():
        if len({int(labels[i]) for i in group}) > 1:
            conflict_groups += 1
            conflict_samples += len(group)
        else:
            keep.append(group[0])
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for c in range(len(CLASSES)):
        idx = np.array([i for i in keep if labels[i] == c], dtype=np.int64)
        rng.shuffle(idx)
        if len(idx) < 2:
            raise ValueError(f'Too few unique images for {CLASSES[c]}')
        n_val = max(1, min(len(idx) - 1, round(len(idx) * val_ratio)))
        val_idx.extend(idx[:n_val].tolist())
        train_idx.extend(idx[n_val:].tolist())
    rng.shuffle(train_idx)
    stats = dict(raw_samples=len(images), unique_pixel_groups=len(groups),
                 conflict_groups=conflict_groups, conflict_samples=conflict_samples,
                 retained_samples=len(keep), train_samples=len(train_idx),
                 validation_samples=len(val_idx))
    return train_idx, val_idx, stats


class Faces(Dataset):
    def __init__(self, images, labels=None, indices=None, size=112, augment=False):
        self.images, self.labels = images, labels
        self.indices = np.arange(len(images)) if indices is None else np.asarray(indices)
        ops = [T.Resize((size, size))]
        if augment:
            ops += [T.RandomHorizontalFlip(),
                    T.RandomAffine(degrees=10, translate=(0.06, 0.06),
                                   scale=(0.9, 1.1), fill=128)]
        ops += [T.Grayscale(num_output_channels=3), T.ToTensor(), T.Normalize(MEAN, STD)]
        if augment:
            ops += [T.RandomErasing(p=0.15, scale=(0.02, 0.10), ratio=(0.5, 2.0))]
        self.transform = T.Compose(ops)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        x = self.transform(Image.fromarray(self.images[idx]))
        return x, int(self.labels[idx]) if self.labels is not None else idx


def make_model(pretrained):
    try:
        model = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
    except Exception as e:
        raise RuntimeError('Pretrained weights could not be loaded. Check network/cache '
                           'or use --no-pretrained to train from scratch.') from e
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    return model


def loader(dataset, args, shuffle=False):
    return DataLoader(dataset, batch_size=args.batch_size, shuffle=shuffle,
                      num_workers=args.workers, pin_memory=args.device == 'cuda',
                      persistent_workers=args.workers > 0)


@torch.inference_mode()
def evaluate(model, batches, device):
    model.eval()
    total_loss, total, correct = 0., 0, 0
    confusion = torch.zeros((len(CLASSES), len(CLASSES)), dtype=torch.long)
    for x, y in batches:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += nn.functional.cross_entropy(logits, y, reduction='sum').item()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
        confusion += torch.bincount((y * len(CLASSES) + pred).cpu(),
                                   minlength=len(CLASSES)**2).reshape(len(CLASSES), -1)
    tp = confusion.diag().float()
    f1 = (2 * tp / (confusion.sum(0) + confusion.sum(1)).clamp_min(1)).mean().item()
    return dict(loss=total_loss / total, accuracy=correct / total, macro_f1=f1,
                confusion_matrix=confusion.tolist())


@torch.inference_mode()
def predict(model, batches, device, tta):
    model.eval()
    results = []
    for x, _ in tqdm(batches, desc='Predict test'):
        x = x.to(device)
        probs = model(x).softmax(dim=1)
        if tta:
            probs = (probs + model(x.flip(-1)).softmax(dim=1)) / 2
        results.append(probs.cpu().numpy())
    return np.concatenate(results)


def write_submission(path, ids, predictions, mapping):
    predictions = np.asarray(predictions)
    if predictions.shape != (len(ids),) or len(ids) != len(set(ids)):
        raise ValueError('Prediction length/shape or ID uniqueness is invalid.')
    if not np.issubdtype(predictions.dtype, np.integer):
        raise ValueError('Predictions must be integer internal class indices.')
    if np.any((predictions < 0) | (predictions >= len(CLASSES))):
        raise ValueError('Predicted class out of range.')
    emotions = [mapping[CLASSES[int(i)]] for i in predictions]
    # csv.writer adds no pandas index column and emits UTF-8 without a BOM.
    with Path(path).open('w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['ID', 'Emotion'])
        writer.writerows(zip(ids, emotions))
    # Re-read the actual file and verify every row before reporting success.
    with Path(path).open(encoding='utf-8', newline='') as f:
        rows = list(csv.reader(f))
    if rows[0] != ['ID', 'Emotion'] or len(rows) != len(ids) + 1:
        raise RuntimeError('Written CSV failed header/row count validation.')
    if [r[0] for r in rows[1:]] != ids:
        raise RuntimeError('Written CSV changed template ID order.')
    if any(len(r) != 2 for r in rows[1:]):
        raise RuntimeError('Written CSV contains an extra or missing column.')
    if [int(r[1]) for r in rows[1:]] != emotions:
        raise RuntimeError('Written CSV changed the predicted submission labels.')
    print(f'Validated submission: {Path(path).resolve()} ({len(ids)} rows)')


def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')


def train(args, images, labels):
    train_idx, val_idx, stats = clean_split(images, labels, args.val_ratio, args.seed)
    print('Data cleaning / split:', json.dumps(stats))
    # Save the exact split for reproducibility. Indices refer to sorted ZIP train members.
    save_json(args.output / 'split.json', dict(train=train_idx, validation=val_idx, **stats))
    train_loader = loader(Faces(images, labels, train_idx, args.image_size, True), args, True)
    val_loader = loader(Faces(images, labels, val_idx, args.image_size), args)
    model = make_model(not args.no_pretrained).to(args.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda', enabled=args.device == 'cuda')
    best_acc, best_loss, stale, history = -1., float('inf'), 0, []
    checkpoint_path = args.output / 'best_model.pt'
    for epoch in range(1, args.epochs + 1):
        start = time.time()
        model.train()
        loss_sum, correct, total = 0., 0, 0
        bar = tqdm(train_loader, desc=f'Epoch {epoch}/{args.epochs}')
        for x, y in bar:
            x, y = x.to(args.device), y.to(args.device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=args.device, enabled=args.device == 'cuda'):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.)
            scaler.step(optimizer)
            scaler.update()
            total += len(y)
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            bar.set_postfix(loss=f'{loss_sum / total:.4f}')
        metrics = evaluate(model, val_loader, args.device)
        record = dict(epoch=epoch, train_loss=loss_sum / total, train_accuracy=correct / total,
                      val_loss=metrics['loss'], val_accuracy=metrics['accuracy'],
                      val_macro_f1=metrics['macro_f1'], lr=optimizer.param_groups[0]['lr'],
                      seconds=round(time.time() - start, 2))
        history.append(record)
        print(json.dumps(record))
        improved = metrics['accuracy'] > best_acc or (
            metrics['accuracy'] == best_acc and metrics['loss'] < best_loss)
        if improved:
            best_acc, best_loss, stale = metrics['accuracy'], metrics['loss'], 0
            torch.save(dict(model_state=model.state_dict(), classes=CLASSES,
                            image_size=args.image_size, epoch=epoch,
                            validation=metrics, seed=args.seed), checkpoint_path)
            save_json(args.output / 'best_validation.json', dict(classes=CLASSES, **metrics))
        else:
            stale += 1
        save_json(args.output / 'history.json', history)
        scheduler.step()
        if stale >= args.patience:
            print(f'Early stopping after {epoch} epochs.')
            break
    return checkpoint_path


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    source = p.add_mutually_exclusive_group()
    source.add_argument('--zip', dest='data', type=Path, help='Original ZIP; auto-discovered on Kaggle if omitted')
    source.add_argument('--data-dir', dest='data', type=Path, help='Extracted dataset folder containing submission.csv')
    p.add_argument('--label-map', type=Path,
                   help='Optional JSON override; defaults to the official IDs embedded in this script')
    p.add_argument('--output', type=Path, default=default_output_dir(),
                   help='Artifact directory (Kaggle: /kaggle/working; local: runs/fer)')
    p.add_argument('--predict-only', type=Path, metavar='CHECKPOINT')
    p.add_argument('--train-only', action='store_true', help='Train without requiring submission IDs')
    p.add_argument('--check-data', action='store_true', help='Validate ZIP and report clean split only')
    p.add_argument('--epochs', type=int, default=35)
    p.add_argument('--batch-size', type=int, default=64)
    p.add_argument('--image-size', type=int, default=112)
    p.add_argument('--lr', type=float, default=3e-4)
    p.add_argument('--weight-decay', type=float, default=1e-4)
    p.add_argument('--val-ratio', type=float, default=0.15)
    p.add_argument('--patience', type=int, default=8)
    p.add_argument('--workers', type=int, default=0, help='0 is portable on Windows; try 2-4 for GPU')
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--threads', type=int, default=4, help='CPU thread count')
    p.add_argument('--device', choices=['auto', 'cpu', 'cuda'], default='auto')
    p.add_argument('--no-pretrained', action='store_true', help='Disable pretrained-weight download')
    p.add_argument('--no-tta', action='store_true', help='Disable horizontal-flip test averaging')
    # Pasted cells have no __file__ and Jupyter adds its own -f kernel.json arguments.
    # Parse only NOTEBOOK_ARGS in a pasted cell. CLI and %run keep their usual arguments.
    if argv is None and '__file__' not in globals():
        argv = globals().get('NOTEBOOK_ARGS', [])
    args = p.parse_args(argv)
    if args.predict_only and args.train_only:
        p.error('--predict-only and --train-only cannot be combined')
    if args.check_data and args.predict_only:
        p.error('--check-data and --predict-only cannot be combined')
    if not 0 < args.val_ratio < 1 or min(args.epochs, args.batch_size, args.patience, args.threads) < 1:
        p.error('epochs/batch-size/patience/threads must be positive; val-ratio must be between 0 and 1')
    if args.workers < 0 or args.image_size < 32 or args.lr <= 0 or args.weight_decay < 0:
        p.error('Invalid workers/image-size/lr/weight-decay')
    if args.data is None:
        args.data = discover_data()
    if not args.data.is_file() and not args.data.is_dir():
        p.error(f'Data path does not exist: {args.data}')
    mapping = None if args.train_only or args.check_data else read_mapping(args.label_map)
    args.device = ('cuda' if torch.cuda.is_available() else 'cpu') if args.device == 'auto' else args.device
    if args.device == 'cuda' and not torch.cuda.is_available():
        p.error('CUDA is unavailable in this PyTorch environment. Install a CUDA build or use --device cpu.')
    torch.set_num_threads(args.threads)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
    args.output.mkdir(parents=True, exist_ok=True)
    print(f'All outputs will be saved to: {args.output.resolve()}')
    save_json(args.output / 'config.json', {k: str(v) if isinstance(v, Path) else v
                                         for k, v in vars(args).items()})
    print(f'Device: {args.device}; internal classes: {CLASSES}')
    print(f'Data source: {args.data.resolve()}')
    images, labels, train_ids, test_images, ids = read_archive(args.data, not bool(args.predict_only))
    print(f'Train: {len(labels)}; test: {len(ids)}; counts: {dict(Counter(labels.tolist()))}')
    if args.check_data:
        _, _, stats = clean_split(images, labels, args.val_ratio, args.seed)
        print(json.dumps(stats, indent=2))
        return
    if not args.predict_only:
        save_json(args.output / 'train_members.json', train_ids)
    checkpoint = args.predict_only or train(args, images, labels)
    if args.train_only:
        print(f'Training complete: {checkpoint.resolve()}')
        print('No submission written. Run --predict-only with the official --label-map when ready.')
        return
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if saved['classes'] != CLASSES:
        raise ValueError('Checkpoint class order differs from this script.')
    model = make_model(False)
    model.load_state_dict(saved['model_state'])
    model.to(args.device)
    print(f'Using best checkpoint epoch={saved["epoch"]}, validation={saved["validation"]["accuracy"]:.4f}')
    probs = predict(model, loader(Faces(test_images, size=saved['image_size']), args),
                    args.device, not args.no_tta)
    write_submission(args.output / 'submission.csv', ids, probs.argmax(axis=1), mapping)
    save_json(args.output / 'label_map_used.json', mapping)
    np.savez_compressed(args.output / 'test_probabilities.npz',
                        ids=np.asarray(ids), classes=np.asarray(CLASSES), probabilities=probs)
    print('Prediction counts:', dict(Counter(CLASSES[i] for i in probs.argmax(axis=1))))


if __name__ == '__main__':
    main()


ValueError: Dataset not found. Add the task data to the Notebook, or specify --zip /path/2026task2.zip or --data-dir /path/to/folder. The extracted folder must contain submission.csv and the train/test trees.

In [3]:
from pathlib import Path
p = Path('/kaggle/input')
print('input exists:', p.exists())
print('top level:', list(p.iterdir()) if p.exists() else [])
for pattern in ('*.zip', '*.csv', 'train', 'test'):
    hits = list(p.rglob(pattern))
    print(pattern, 'count=', len(hits), 'examples=', [str(x) for x in hits[:20]])

input exists: True
top level: []
*.zip count= 0 examples= []
*.csv count= 0 examples= []
train count= 0 examples= []
test count= 0 examples= []
